# Colab runner

This notebook only mounts Drive, pulls the repository, installs the package, and starts scripts.
All logic lives in the package `ddd` and in `scripts/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # needs a click by the owner

In [ ]:
REPO_URL = 'https://github.com/SauHin/driver-distraction.git'
REPO = '/content/driver-distraction'
DRIVE = '/content/drive/MyDrive/driver-distraction'  # data, manifests, logs, runs
%env REPO={REPO}
%env DRIVE={DRIVE}

In [ ]:
%%bash
if [ -d "$REPO/.git" ]; then git -C "$REPO" pull --ff-only; else git clone "$REPO_URL" "$REPO"; fi
git -C "$REPO" log -1 --oneline

In [ ]:
%%bash
pip install -q -e "$REPO[test]"
cd "$REPO" && python -m pytest -q

In [ ]:
%%bash
# Copy the Kaggle zip from Drive to the local disk, unzip it, then compare with the manifest of the laptop.
mkdir -p "$REPO/data/raw/statefarm"
cp "$DRIVE/data/raw/statefarm/state-farm-distracted-driver-detection.zip" "$REPO/data/raw/statefarm/"
cd "$REPO/data/raw/statefarm" && unzip -q -o state-farm-distracted-driver-detection.zip
cd "$REPO" && python scripts/make_manifest.py data/raw/statefarm "$DRIVE/manifests/statefarm.csv" --check

In [ ]:
%%bash
# Start a long job in the background. The log goes to Drive. Replace the script for each job.
mkdir -p "$DRIVE/logs"
cd "$REPO" && nohup python scripts/audit_statefarm.py --out "$DRIVE/logs/statefarm_audit.json"   > "$DRIVE/logs/audit.log" 2>&1 &
echo "started, log: $DRIVE/logs/audit.log"

In [ ]:
!tail -n 20 "$DRIVE/logs/audit.log"